<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 12 · El primer modelo: regresión lineal simple

La semana pasada quedó instalada la regla que gobierna todo lo que viene: un modelo solo existe si le
gana a su línea base, y solo vale la pena si la diferencia paga lo que cuesta construirlo. Hoy
construyes el primero. Una variable explica otra, se ajusta una recta y **el coeficiente se lee en
dólares y unidades, no en unidades de estadística**. Empiezas con Advertising, el conjunto canónico con
el que se enseña esto desde hace treinta años, y después lo aplicas al negocio del curso: la inversión
mensual en volantes de Comercial Andina contra sus ventas del mes. El segundo caso trae una sorpresa
que la semana 9 te dejó preparada para ver.

> **Hoy haces** · Ajustas una regresión lineal simple con scikit-learn y con statsmodels sobre
> Advertising, lees el coeficiente en dólares y decides si el anuncio se paga solo (90 min). Repites
> sobre `marketing_mensual.csv` de Comercial Andina, apartas el conjunto de prueba, comparas contra
> `DummyRegressor` y reportas el error en unidades de negocio: error absoluto medio y error porcentual.
> Cierras con el cálculo que ninguna métrica estándar hace: cuánto cuesta equivocarse por debajo frente
> a equivocarse por arriba, y cuánto sesgo conviene meterle al pronóstico.
>
> **Entrega** · Este cuaderno ejecutado, el modelo del negocio del caso con su línea base declarada y
> su error expresado en unidades de negocio, la interpretación del coeficiente en una frase que un
> gerente entienda, y el cálculo del costo asimétrico del error con sus dos supuestos escritos.
> Nombre de archivo: `lab_12_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              Path("/content/CursoAnalisisDatos_IA_2026/sitio/datos")]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    raise FileNotFoundError(
        "No encuentro la carpeta de datos. En Colab ejecuta primero:\n"
        "  !git clone https://github.com/<usuario>/CursoAnalisisDatos_IA_2026.git")

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. Advertising: doscientos mercados y una pregunta

`Advertising.csv` tiene doscientas filas. Cada una es un mercado: cuánto se invirtió en televisión,
radio y prensa —en miles de dólares— y cuántas unidades se vendieron —en miles—. La pregunta de negocio
cabe en una línea: **si pongo mil dólares más en televisión, ¿cuánto vendo?**

Se carga por URL, así que funciona igual en Colab que en tu máquina.

In [ ]:
URL_ADV = ("https://raw.githubusercontent.com/justmarkham/scikit-learn-videos/"
           "master/data/Advertising.csv")
adv = pd.read_csv(URL_ADV, index_col=0)

print(f"{adv.shape[0]} mercados × {adv.shape[1]} columnas · "
      f"TV, Radio y Newspaper en miles de dólares · Sales en miles de unidades\n")
print(adv.describe().round(2).to_string())
print(f"\ncorrelación de cada medio con las ventas:")
print(adv.corr()["Sales"].drop("Sales").to_string(float_format=lambda v: f"{v:.4f}"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, medio in zip(axes, ["TV", "Radio", "Newspaper"]):
    ax.scatter(adv[medio], adv["Sales"], s=14, alpha=0.6, color="#4C72B0")
    ax.set_xlabel(f"{medio} (miles de dólares)")
    ax.set_title(f"{medio} · correlación {adv[medio].corr(adv['Sales']):.3f}", fontsize=11)
axes[0].set_ylabel("Sales (miles de unidades)")
fig.suptitle("Televisión dibuja una relación clara; prensa es una nube sin forma",
             fontsize=12, y=1.04)
plt.tight_layout()
plt.show()

Televisión correlaciona 0,7822 con las ventas, radio 0,5762 y prensa 0,2283. El gráfico ya dice
casi todo: en el panel de TV hay una tendencia que se puede seguir con el dedo, y en el de prensa hay
una nube. Lo que el gráfico **no** dice es cuánto vale un dólar de televisión, y para eso hace falta
ajustar la recta.

## 2. Mínimos cuadrados: qué se minimiza exactamente

Entre todas las rectas posibles, la regresión elige la que hace más pequeña la **suma de los residuos
al cuadrado**. El residuo es la distancia vertical entre el punto real y la recta: lo que el modelo se
equivocó en ese mercado. Se elevan al cuadrado por dos razones —para que los errores por arriba no
cancelen los de abajo, y para castigar más los errores grandes— y esa segunda decisión tiene
consecuencias de negocio que veremos en la sección 6.

In [ ]:
def suma_de_cuadrados(pendiente, intercepto, x, y):
    return ((y - (intercepto + pendiente * x)) ** 2).sum()


x, y = adv["TV"].values, adv["Sales"].values
candidatas = pd.DataFrame({"pendiente": [0.030, 0.040, 0.045, 0.047537, 0.050, 0.060]})
candidatas["intercepto"] = 7.032594
candidatas["suma de residuos²"] = [suma_de_cuadrados(p, 7.032594, x, y)
                                   for p in candidatas["pendiente"]]
candidatas["peor que la mejor en"] = (candidatas["suma de residuos²"]
                                      - candidatas["suma de residuos²"].min())
print("La misma recta con seis pendientes distintas:\n")
print(candidatas.to_string(index=False, float_format=lambda v: f"{v:,.6f}"))
print(f"\nLa mejor de las seis es {candidatas.loc[candidatas['suma de residuos²'].idxmin(), 'pendiente']}, "
      "y no es casualidad: es la que calcula la fórmula de mínimos cuadrados.")
print(f"pendiente por fórmula = covarianza / varianza = "
      f"{np.cov(x, y)[0, 1] / np.var(x, ddof=1):.6f}")

In [ ]:
from sklearn.linear_model import LinearRegression

X = adv[["TV"]]
modelo_tv = LinearRegression().fit(X, adv["Sales"])
b1, b0 = modelo_tv.coef_[0], modelo_tv.intercept_

print(f"pendiente (coeficiente de TV) : {b1:.6f}")
print(f"intercepto                    : {b0:.6f}")
print(f"R²                            : {modelo_tv.score(X, adv['Sales']):.6f}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4))
rejilla = np.linspace(adv["TV"].min(), adv["TV"].max(), 100)
axes[0].scatter(adv["TV"], adv["Sales"], s=16, alpha=0.5, color="#4C72B0")
axes[0].plot(rejilla, b0 + b1 * rejilla, color="#C44E52", linewidth=2)
axes[0].set_xlabel("inversión en TV (miles de dólares)")
axes[0].set_ylabel("ventas (miles de unidades)")
axes[0].set_title(f"Cada mil dólares en TV mueven {b1 * 1000:.1f} unidades", fontsize=11)

residuos = adv["Sales"] - modelo_tv.predict(X)
axes[1].scatter(modelo_tv.predict(X), residuos, s=16, alpha=0.5, color="#55A868")
axes[1].axhline(0, color="#C44E52", linewidth=1.5)
axes[1].set_xlabel("ventas pronosticadas")
axes[1].set_ylabel("residuo (real − pronóstico)")
axes[1].set_title("Los residuos se abren a la derecha: el error crece con la inversión",
                  fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nresiduo mayor: {residuos.max():.2f} · residuo menor: {residuos.min():.2f} "
      f"· suma de residuos: {residuos.sum():.10f}")

### Cómo se lee el coeficiente

El número que sale de scikit-learn es 0,047537. Esa cifra, tal cual, no le sirve a nadie. Traducida:

> **Cada mil dólares adicionales de inversión en televisión se asocian con 47,5 unidades más vendidas.**

Y ahí es donde empieza la decisión de negocio, no donde termina. Para que esos mil dólares se paguen
solos, cada unidad tiene que dejar al menos **21,04 dólares de margen** —mil dividido entre 47,5—. Si
el producto deja 5 dólares, el anuncio pierde 762 dólares por cada mil invertidos; si deja 30, gana 426.
**El coeficiente no dice si hay que anunciar: dice a partir de qué margen conviene anunciar.**

El intercepto es 7,03: las ventas que el modelo predice con cero inversión en televisión. Se puede
interpretar aquí porque hay mercados con inversión cercana a cero en los datos. Cuando no los hay, el
intercepto es solo el punto donde la recta cruza el eje y no significa nada.

⚠️ El gráfico de residuos ya avisa de algo: se abren en forma de embudo hacia la derecha. El modelo se
equivoca poco en los mercados pequeños y mucho en los grandes, así que **el error medio que vamos a
calcular en la sección 4 no reparte parejo**. Ese diagnóstico completo es la semana 13.

## 3. Las tres variables, comparadas

Repetir el ajuste con cada medio contesta la pregunta que el gerente de marketing va a hacer primero:
¿cuál explica más por sí sola?

In [ ]:
comparacion = []
for medio in ["TV", "Radio", "Newspaper"]:
    m = LinearRegression().fit(adv[[medio]], adv["Sales"])
    comparacion.append((medio, m.coef_[0], m.intercept_, m.score(adv[[medio]], adv["Sales"]),
                        m.coef_[0] * 1000, 1000 / (m.coef_[0] * 1000)))
comparacion = pd.DataFrame(comparacion, columns=[
    "medio", "coeficiente", "intercepto", "R²", "unidades por cada 1 000 dólares",
    "margen mínimo por unidad para que se pague"])
print(comparacion.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
print("\nRadio mueve más unidades por dólar que televisión (202,5 contra 47,5) y explica")
print("la mitad de la variación (R² 0,33 contra 0,61). Son dos preguntas distintas:")
print("  · el coeficiente responde cuánto rinde un dólar")
print("  · el R² responde cuánto de lo que pasa se explica con esta sola variable")

📌 **Radio rinde cuatro veces más por dólar que televisión y explica la mitad.** Con el coeficiente
en la mano, mil dólares en radio mueven 202,5 unidades contra 47,5 de televisión, así que el margen
mínimo para que el anuncio se pague baja de 21,04 a 4,94 dólares por unidad. Con el R² en la mano,
televisión gana. Las dos lecturas son correctas y contestan preguntas distintas, y confundirlas es el
origen de la mitad de las discusiones de presupuesto de marketing.

## 4. La línea base obligatoria

Antes de creerse nada: se aparta el conjunto de prueba, se calcula lo que hace `DummyRegressor` sin
aprender, y solo entonces se mira el modelo. La regla de la semana 11 aplicada a la regresión.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score


def evaluar(y_real, y_pred, nombre):
    return pd.Series({
        "error absoluto medio": mean_absolute_error(y_real, y_pred),
        "error porcentual medio": np.mean(np.abs((y_real - y_pred) / y_real)) * 100,
        "R²": r2_score(y_real, y_pred),
    }, name=nombre)


X_ent, X_pru, y_ent, y_pru = train_test_split(adv[["TV"]], adv["Sales"],
                                              test_size=0.25, random_state=SEED)
base = DummyRegressor(strategy="mean").fit(X_ent, y_ent)
mod = LinearRegression().fit(X_ent, y_ent)

tabla = pd.concat([evaluar(y_pru, base.predict(X_pru), "línea base (la media)"),
                   evaluar(y_pru, mod.predict(X_pru), "regresión sobre TV")], axis=1)
tabla["mejora"] = tabla["línea base (la media)"] - tabla["regresión sobre TV"]

print(f"entrenamiento {len(X_ent)} mercados · prueba {len(X_pru)} mercados")
print(f"la línea base predice siempre {base.predict(X_pru)[0]:.4f} "
      f"(la media de las ventas del entrenamiento)\n")
print(tabla.to_string(float_format=lambda v: f"{v:,.4f}"))
mae_base = mean_absolute_error(y_pru, base.predict(X_pru))
mae_mod = mean_absolute_error(y_pru, mod.predict(X_pru))
print(f"\nel modelo reduce el error absoluto medio un {1 - mae_mod / mae_base:.2%} "
      f"({mae_base:.4f} → {mae_mod:.4f})")

El modelo **le gana a la línea base y por mucho**: reduce el error absoluto medio un 51,39 %, de
4,6777 a 2,2738 miles de unidades. Esa es la frase que va al informe, y va con los dos números, no con
uno. Un error de 2,27 sin el 4,68 al lado no significa nada.

Fíjate también en el R² de la línea base: **−0,0471**. Un R² negativo no es un error de cálculo,
significa que el modelo predice peor que la media del conjunto de prueba. `DummyRegressor` predice la
media del **entrenamiento**, que no es exactamente la del conjunto de prueba, y ahí nace ese signo.

## 5. Comercial Andina: los volantes y las ventas del mes

Ahora el negocio del curso. `marketing_mensual.csv` tiene una fila por mes con lo invertido en radio,
digital y volantes, y lo facturado. La pregunta que llega del gerente: **¿cuánto vende un dólar de
volantes?**

In [ ]:
marketing = pd.read_csv(DATOS / "marketing_mensual.csv", parse_dates=["mes"])
print(f"{len(marketing)} meses · de {marketing['mes'].min():%m-%Y} a {marketing['mes'].max():%m-%Y}\n")
print(marketing.tail(3).to_string(index=False))

# El último mes no está cerrado: solo tiene notas de crédito de ventas anteriores.
mk = marketing[marketing["ventas_mes"] > 0].copy()
print(f"\nmeses descartados por no estar cerrados: {len(marketing) - len(mk)} "
      f"(ventas_mes = {marketing.loc[marketing['ventas_mes'] <= 0, 'ventas_mes'].iloc[0]:,.2f})")
print(f"meses utilizables: {len(mk)}\n")
print("correlación de cada medio con las ventas del mes:")
print(mk[["inversion_radio", "inversion_digital", "inversion_volantes"]]
      .corrwith(mk["ventas_mes"]).to_string(float_format=lambda v: f"{v:+.4f}"))

⚠️ **El último mes tiene facturación negativa y hay que sacarlo.** Julio de 2026 no está cerrado:
lo único que hay registrado son notas de crédito de ventas de junio, así que sus ventas suman −776,80.
Dejarlo dentro arrastraría la recta hacia abajo con un punto que no representa ningún mes real. Esta es
la decisión de la semana 4 —qué es un dato y qué es un artefacto del corte— apareciendo dentro de un
modelo. Se declara y se sigue con 30 meses.

Los volantes correlacionan +0,9397 con las ventas del mes. Radio se queda en −0,0292 y digital en
−0,2501: ninguno de los dos sirve para pronosticar, y el signo negativo de digital es una advertencia,
no un hallazgo —con treinta puntos, una correlación de esa magnitud es indistinguible del ruido—.
Ajustamos con volantes.

In [ ]:
import statsmodels.api as sm

Xa, ya = mk[["inversion_volantes"]], mk["ventas_mes"]
lineal = LinearRegression().fit(Xa, ya)
ols = sm.OLS(ya, sm.add_constant(Xa)).fit()

print(f"scikit-learn · coeficiente {lineal.coef_[0]:,.6f} · intercepto {lineal.intercept_:,.4f} "
      f"· R² {lineal.score(Xa, ya):.6f}\n")
print(ols.summary().tables[1])
print(f"\np-valor de la pendiente : {ols.pvalues.iloc[1]:.3g}")
print(f"intervalo de confianza  : [{ols.conf_int().iloc[1, 0]:,.2f} , "
      f"{ols.conf_int().iloc[1, 1]:,.2f}] dólares de venta por dólar de volante")

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(mk["inversion_volantes"], mk["ventas_mes"], s=45, color="#4C72B0")
rej = np.linspace(mk["inversion_volantes"].min(), mk["inversion_volantes"].max(), 50)
ax.plot(rej, lineal.intercept_ + lineal.coef_[0] * rej, color="#C44E52", linewidth=2)
ax.set_xlabel("inversión en volantes del mes (dólares)")
ax.set_ylabel("ventas del mes (dólares)")
ax.set_title(f"Cada dólar de volantes acompaña a {lineal.coef_[0]:,.2f} dólares de venta "
             f"(R² = {lineal.score(Xa, ya):.4f})")
plt.tight_layout()
plt.show()

### La lectura del coeficiente, y por qué no es la que parece

El coeficiente es 38,34 con un p-valor de 1,42e-14 y un intervalo de confianza de [32,94 , 43,74]. La
lectura estadística es impecable. La lectura ingenua de negocio sería:

> «Cada dólar invertido en volantes produce 38,34 dólares de venta. Retorno del 3 734 %.»

**Y es falsa.** Si eso fuera cierto, Comercial Andina debería invertir todo su capital en volantes y
retirarse. La semana 9 dejó la herramienta para ver por qué: **causalidad inversa**. En esta empresa el
presupuesto de volantes de cada mes se fija en función de las ventas que se esperan —más presupuesto en
diciembre, menos en febrero— así que la flecha va de las ventas al presupuesto, no al revés. El modelo
está midiendo cómo se reparte el presupuesto, no qué produce el volante.

Eso **no** invalida el modelo, cambia para qué sirve:

- ✅ **Sirve para pronosticar.** Dado el presupuesto que el área de marketing ya decidió para el mes que
  viene, predice las ventas con un error que vamos a medir ahora.
- ❌ **No sirve para decidir el presupuesto.** Para eso hace falta una prueba A/B: repartir volantes en
  unas zonas y no en otras, aleatorizando, exactamente como en la semana 9.

La frase que va al informe es la primera, nunca la segunda. Y va con la limitación escrita al lado.

In [ ]:
Xa_ent, Xa_pru, ya_ent, ya_pru = train_test_split(Xa, ya, test_size=0.25, random_state=SEED)
base_a = DummyRegressor(strategy="mean").fit(Xa_ent, ya_ent)
mod_a = LinearRegression().fit(Xa_ent, ya_ent)
pred_a = mod_a.predict(Xa_pru)

tabla_a = pd.concat([evaluar(ya_pru, base_a.predict(Xa_pru), "línea base (la media)"),
                     evaluar(ya_pru, pred_a, "regresión sobre volantes")], axis=1)
print(f"entrenamiento {len(Xa_ent)} meses · prueba {len(Xa_pru)} meses")
print(f"la línea base predice siempre {base_a.predict(Xa_pru)[0]:,.2f} dólares al mes\n")
print(tabla_a.to_string(float_format=lambda v: f"{v:,.4f}"))

mae_b = mean_absolute_error(ya_pru, base_a.predict(Xa_pru))
mae_m = mean_absolute_error(ya_pru, pred_a)
print(f"\nmejora del error absoluto medio: {1 - mae_m / mae_b:.2%} "
      f"({mae_b:,.2f} → {mae_m:,.2f} dólares al mes)")
print(f"error porcentual: {np.mean(np.abs((ya_pru - base_a.predict(Xa_pru)) / ya_pru)):.2%} "
      f"→ {np.mean(np.abs((ya_pru - pred_a) / ya_pru)):.2%}")

detalle = pd.DataFrame({"ventas reales": ya_pru.values, "pronóstico": pred_a})
detalle["error"] = detalle["ventas reales"] - detalle["pronóstico"]
detalle["error %"] = detalle["error"] / detalle["ventas reales"] * 100
print("\nMes a mes, los ocho meses del conjunto de prueba:")
print(detalle.round(2).to_string(index=False))

📌 **El modelo reduce el error de 15 792,92 a 4 711,95 dólares al mes: un 70,16 %.** En porcentaje
sobre las ventas, el error baja del 14,04 % al 4,90 %. Esa es la frase completa para el informe:
*«el pronóstico de ventas mensuales tiene un error medio del 4,90 %, frente al 14,04 % de la regla
actual de usar el promedio»*.

Y ahora la parte que ninguna métrica de scikit-learn contesta: **de los ocho meses, en cinco el modelo
se quedó corto y en tres se pasó.** ¿Da igual? No.

## 6. El error de pronóstico no es simétrico

El error absoluto medio trata igual quedarse corto que pasarse, y el negocio no. Si el pronóstico se
queda corto, Comercial Andina no compró suficiente mercadería y **pierde el margen de las ventas que no
pudo hacer**. Si se pasa, compró de más y **paga el costo financiero y de bodega del inventario que
sobró**. Los dos números son distintos y hay que escribirlos antes de elegir el pronóstico.

In [ ]:
# --- Supuestos declarados. Los dos salen de la contabilidad, no del modelo. ---
productos = pd.read_csv(DATOS / "productos.csv")
ventas = pd.read_csv(DATOS / "ventas_limpias.csv", parse_dates=["fecha"])
v = ventas.merge(productos[["producto_id", "costo_unitario"]], on="producto_id",
                 how="left", validate="m:1")
v["monto"] = v["cantidad"] * v["precio_unitario"] * (1 - v["descuento"])
v["margen"] = v["monto"] - v["cantidad"] * v["costo_unitario"]

MARGEN = v["margen"].sum() / v["monto"].sum()   # lo que se pierde por cada dólar no vendido
COSTO_EXCESO = 0.08                             # costo financiero y de bodega del inventario sobrante

print(f"margen sobre ventas de Comercial Andina : {MARGEN:.2%}  ← se pierde si falta producto")
print(f"costo del inventario que sobra          : {COSTO_EXCESO:.2%} sobre el costo de la mercadería\n")


def costo_del_error(real, pronostico, margen=MARGEN, exceso=COSTO_EXCESO):
    """Quedarse corto cuesta el margen perdido; pasarse cuesta mantener el inventario."""
    falta = np.clip(real - pronostico, 0, None)
    sobra = np.clip(pronostico - real, 0, None)
    return falta * margen + sobra * exceso * (1 - margen)


detalle["costo del error"] = costo_del_error(detalle["ventas reales"], detalle["pronóstico"])
detalle["tipo"] = np.where(detalle["error"] > 0, "faltó producto", "sobró producto")
print(detalle[["ventas reales", "pronóstico", "error", "tipo", "costo del error"]]
      .round(2).to_string(index=False))
print(f"\ncosto total de los ocho meses : {detalle['costo del error'].sum():,.2f}")
print(f"quedarse corto cuesta {MARGEN / (COSTO_EXCESO * (1 - MARGEN)):.1f} veces más "
      f"que pasarse, por cada dólar de desvío")
ratio_critico = MARGEN / (MARGEN + COSTO_EXCESO * (1 - MARGEN))
print(f"ratio crítico = {MARGEN:.4f} / ({MARGEN:.4f} + {COSTO_EXCESO * (1 - MARGEN):.4f}) "
      f"= {ratio_critico:.4f}  → conviene planificar para el percentil {ratio_critico * 100:.0f}")

In [ ]:
sesgos = pd.DataFrame({"sesgo": np.arange(0, 0.31, 0.025)})
sesgos["costo"] = [costo_del_error(detalle["ventas reales"], detalle["pronóstico"] * (1 + s)).sum()
                   for s in sesgos["sesgo"]]
sesgos["error absoluto medio"] = [
    mean_absolute_error(detalle["ventas reales"], detalle["pronóstico"] * (1 + s))
    for s in sesgos["sesgo"]]
mejor = sesgos.loc[sesgos["costo"].idxmin()]

print(sesgos.round(2).to_string(index=False))
print(f"\nsesgo que minimiza el COSTO  : +{mejor['sesgo']:.1%}  → {mejor['costo']:,.2f}")
print(f"sesgo que minimiza el ERROR  : +{sesgos.loc[sesgos['error absoluto medio'].idxmin(), 'sesgo']:.1%}"
      f"  → {sesgos['error absoluto medio'].min():,.2f}")
print(f"ahorro de pronosticar por lo alto: "
      f"{sesgos.loc[0, 'costo'] - mejor['costo']:,.2f} "
      f"({1 - mejor['costo'] / sesgos.loc[0, 'costo']:.1%} menos)")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sesgos["sesgo"] * 100, sesgos["costo"], marker="o", color="#C44E52",
        label="costo del error en dólares")
ax.plot(sesgos["sesgo"] * 100, sesgos["error absoluto medio"], marker="o", color="#4C72B0",
        label="error absoluto medio")
ax.axvline(mejor["sesgo"] * 100, color="#55A868", linestyle="--", label="mínimo costo")
ax.set_xlabel("sesgo aplicado al pronóstico (%)")
ax.set_ylabel("dólares")
ax.set_title("El pronóstico más exacto no es el más barato: conviene pedir un 7,5 % de más")
ax.legend()
plt.tight_layout()
plt.show()

📌 **El pronóstico más exacto cuesta 8 726,75 dólares y el más barato cuesta 3 256,28.** El segundo
tiene un error absoluto medio de 8 152,85 —un 73 % peor que los 4 711,95 del pronóstico neutro— y aun
así ahorra el 62,7 % del dinero. La razón está en la primera tabla: en Comercial Andina quedarse sin
producto cuesta **9,9 veces más** que que sobre, porque se pierde el 44,09 % de margen contra un 4,47 %
de costo de bodega sobre el precio de venta.

Eso tiene nombre y fórmula. La proporción de la demanda que conviene cubrir es el **ratio crítico**:

> ratio crítico = costo de faltar ÷ (costo de faltar + costo de sobrar) = 0,4409 ÷ (0,4409 + 0,0447) = **0,9079**


Es decir: conviene planificar para cubrir el percentil 91 de la demanda, no la media. **La media es la
respuesta correcta solo cuando los dos errores cuestan lo mismo, y casi nunca cuestan lo mismo.**

La consecuencia práctica para el entregable: el modelo se reporta con su error absoluto medio **y** con
el costo en dólares de ese error, y si los dos apuntan a pronósticos distintos, manda el dinero.

### 🌶️ Ejercicio 1 — Guiado

Repite el ajuste de Advertising con **Radio** en lugar de TV y compáralos honestamente: coeficiente,
R², error absoluto medio y error porcentual sobre el mismo conjunto de prueba, y la línea base al lado.
Después contesta con números: si el producto deja 8 dólares de margen por unidad, ¿en qué medio conviene
poner los próximos mil dólares?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: reutiliza la función evaluar() y el mismo random_state=SEED para que la comparación sea justa
# Pista 2: el margen mínimo por unidad para que el anuncio se pague es 1000 / (coef * 1000)
# Pista 3: cuidado con concluir a partir del R² solo. Reporta las cuatro métricas y di cuál usas
#          para decidir y por qué

### 🔥 Desafío · dividido en dos

Construye el pronóstico mensual de Comercial Andina **por ciudad** en lugar de para el total.

**Parte A · en clase (20 min).** Arma la tabla de ventas mensuales por ciudad desde `ventas_limpias.csv`
y `sucursales.csv`, y ajusta el modelo **de una sola ciudad**: la que más factura. Repórtalo contra su
propia línea base con el error absoluto medio de las dos, igual que en la sección 3. Una ciudad basta
para ver si el método se sostiene.

**Parte B · para casa (30 min).** Repite el ajuste en las cinco ciudades restantes, explica por qué el
modelo funciona mucho peor en unas que en otras y cierra con la recomendación: ¿en cuáles se usa el
modelo y en cuáles se sigue con la media? Va en la entrega del cuaderno.

In [ ]:
# TU CÓDIGO AQUÍ
# --- Parte A (en clase): una sola ciudad ---
# Pista 1: la ciudad de la venta viene de sucursales.csv, no de clientes.csv. El canal en línea
#          aparece como ciudad "Nacional" y hay que decidir qué se hace con él
# Pista 2: con 30 meses por ciudad, el conjunto de prueba queda en 7 u 8 meses: dilo en las
#          limitaciones, porque un error medido sobre 8 puntos es muy inestable
#
# --- Parte B (para casa): las demás ciudades y la recomendación ---
# Pista 3: la inversión en volantes es NACIONAL, la misma para las cinco ciudades. Piensa qué
#          significa eso para el coeficiente de cada ciudad antes de interpretarlo

### 🎯 Reto en clase (15 min)

En equipos y contra reloj: **la subasta del coeficiente**. El docente proyecta un coeficiente de una
regresión real sin decir de qué son las variables —«el coeficiente es 0,047»— y cada equipo tiene tres
minutos para escribir la frase de negocio completa: qué sube, cuánto, en qué unidades y qué decisión
cambia. Después se revelan las variables y se comparan las frases. Se puntúa la que un gerente podría
leer en voz alta en una reunión sin traducir nada. Las frases que empiezan con «por cada unidad de
aumento en la variable independiente» valen cero.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: el molde de la frase tiene cuatro ranuras y las cuatro son obligatorias.
#   "Cada <cantidad + unidad de la X> adicional se asocia con <coef * cantidad> <unidad de la Y> más,
#    lo que significa que <decisión> conviene si <umbral calculado>."
# Comprueba tu frase con el modelo de TV: cantidad = 1000 dólares, coef * 1000 = 47,5 unidades,
# decisión = anunciar, umbral = margen por unidad mayor que 21,04

## La trampa de hoy

⚠️ **Reportar el R² como si fuera la calidad del modelo.** El R² dice qué proporción de la variación de
la Y explica la X, y eso depende tanto del modelo como de **cuánto varía la Y**. Un R² alto sobre una
variable muy dispersa puede convivir con un error porcentual que hace imposible planificar nada.

Cuatro modelos reales, tres de ellos ya ajustados en este cuaderno, ordenados de las dos maneras.

In [ ]:
compras = v[~v["es_devolucion"]]
por_factura = compras.groupby("factura_id").agg(unidades=("cantidad", "sum"),
                                                monto=("monto", "sum")).reset_index()
Xf_ent, Xf_pru, yf_ent, yf_pru = train_test_split(por_factura[["unidades"]], por_factura["monto"],
                                                  test_size=0.25, random_state=SEED)
mod_f = LinearRegression().fit(Xf_ent, yf_ent)

Xn_ent, Xn_pru, yn_ent, yn_pru = train_test_split(adv[["Newspaper"]], adv["Sales"],
                                                  test_size=0.25, random_state=SEED)
mod_n = LinearRegression().fit(Xn_ent, yn_ent)

modelos = pd.concat([
    evaluar(yf_pru, mod_f.predict(Xf_pru), "Andina · monto de la factura ~ unidades"),
    evaluar(ya_pru, pred_a, "Andina · ventas del mes ~ volantes"),
    evaluar(y_pru, mod.predict(X_pru), "Advertising · Sales ~ TV"),
    evaluar(yn_pru, mod_n.predict(Xn_pru), "Advertising · Sales ~ Newspaper"),
], axis=1).T

TOLERANCIA = 10.0   # el negocio puede planificar si el error porcentual es menor que esto
modelos["¿sirve para planificar?"] = np.where(
    modelos["error porcentual medio"] < TOLERANCIA, "SÍ", "NO")

print("Ordenados por R², que es como se presentan siempre:\n")
print(modelos.sort_values("R²", ascending=False).to_string(float_format=lambda v: f"{v:,.4f}"))
print("\nOrdenados por error porcentual, que es lo que necesita quien planifica inventario:\n")
print(modelos.sort_values("error porcentual medio").to_string(float_format=lambda v: f"{v:,.4f}"))

📌 **El modelo con el R² más alto de los cuatro es el segundo peor para planificar.** «Monto de la
factura a partir de las unidades» tiene un R² de 0,9197 —el mejor de la tabla, el número que cualquiera
llevaría a una presentación— y un error porcentual del 29,87 %. Con ese modelo, la estimación de lo que
va a dejar una visita se equivoca en casi un tercio: no sirve para planificar caja, ni personal, ni
inventario.

Y al revés: el modelo de Advertising sobre TV, con un R² de 0,6606 que se considera respetable, falla
un 17,92 %. El único de los cuatro que pasa la tolerancia del 10 % es el de los volantes, con un R² de
0,8868 que ni siquiera es el más alto.

**Los dos ordenamientos no coinciden, y solo uno de los dos contesta la pregunta del negocio.** La
razón es que el R² se mide contra la varianza de la variable objetivo: cuanto más dispersa está la Y,
más fácil es explicar «mucha» variación y menos significa hacerlo. El error porcentual se mide contra el
tamaño de la Y, que es lo que le importa a quien tiene que comprar mercadería.

La regla del curso, sin excepciones: **ningún modelo se reporta solo con el R².** Va siempre con cuatro
cosas al lado —el error en unidades de negocio, el error porcentual, la línea base y el costo en dólares
de equivocarse— y la decisión de usarlo o no se toma con una tolerancia escrita **antes** de ver los
resultados. Aquí la tolerancia era el 10 % y la escribimos antes: por eso la tabla se puede leer sin
discutir.

## Entregable

Sube `lab_12_apellido.ipynb` con:

- El modelo de Advertising ajustado con scikit-learn, con el coeficiente **traducido a la frase de
  negocio**: mil dólares en televisión mueven 47,5 unidades, y el anuncio se paga si cada unidad deja
  más de 21,04 dólares de margen.
- El modelo de Comercial Andina con `statsmodels`, con el p-valor y el intervalo de confianza del
  coeficiente, y la limitación de causalidad inversa escrita: sirve para pronosticar, no para decidir
  el presupuesto.
- Los dos modelos comparados contra `DummyRegressor` con el conjunto de prueba apartado, y la mejora
  expresada en las dos cifras: 4,6777 → 2,2738 en Advertising (51,39 %) y 15 792,92 → 4 711,95 en
  Comercial Andina (70,16 %).
- El **costo asimétrico del error** con sus dos supuestos declarados —44,09 % de margen perdido si
  falta, 8 % de costo de inventario si sobra— y el sesgo que minimiza el costo, con la diferencia
  respecto del que minimiza el error.
- La tabla de los cuatro modelos ordenada de las dos formas, con la tolerancia del negocio escrita
  antes de mirarla.
- Una fila nueva en la bitácora de prompts: le pediste al asistente que interpretara tu coeficiente.
  Deja **las dos versiones** en el cuaderno, la estadística que devolvió y la de negocio que
  reescribiste tú, y una línea sobre qué le faltaba a la primera.

## Para tu equipo

- El pronóstico del negocio del caso se entrega **con la línea base declarada**, no sin ella. Si el
  modelo no le gana a la media o al mismo mes del año pasado, esa es la conclusión y se escribe: es un
  resultado, no un fracaso.
- El error se expresa en unidades del negocio —cajas, dólares, clientes, horas de personal— y nunca solo
  en R². Si el gerente no puede decir «con ese error puedo o no puedo trabajar», la métrica está mal
  elegida.
- Pregunten en la empresa cuánto cuesta quedarse sin producto y cuánto cuesta que sobre. Casi nunca
  tienen los dos números y casi siempre tienen una intuición fuerte sobre cuál es peor. Esa
  conversación vale más que afinar el modelo dos décimas de R².